## 03b_kpi_aggregations — RCM Intelligence Platform
### Purpose
Builds all Gold KPI aggregation tables from the fact table.
These tables power every dashboard widget and Genie query.

### KPI tables built
| Table | Description |
|---|---|
| kpi_by_state | Revenue gap, CTP ratio, underpayment rate by state |
| kpi_by_drg | Top DRGs by revenue leakage nationally |
| ar_aging_buckets | Simulated AR aging by payment lag buckets |
| hospital_scorecard | Per-hospital RCM health score (0-100) |

### Inputs
- rcm_platform.rcm_gold.fact_claims

### Outputs
- rcm_platform.rcm_gold.kpi_by_state
- rcm_platform.rcm_gold.kpi_by_drg
- rcm_platform.rcm_gold.ar_aging_buckets
- rcm_platform.rcm_gold.hospital_scorecard

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Gold |
| Run order  | 7 — after 03a_fact_claims |
| Depends on | 00_config, 00_utils |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql.functions import (
    col, lit, when, round, expr, sum, avg,
    count, countDistinct, max, min, percent_rank
)
from pyspark.sql.window import Window

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "03b_kpi_aggregations"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")

# Read fact table once — reuse for all aggregations
df_fact = spark.table(TBL_FACT_CLAIMS)
fact_count = df_fact.count()
print(f"Fact table rows : {fact_count:,}")

In [0]:
# ============================================================
# STEP 1 — KPI BY STATE
# State-level RCM performance metrics
# Used for the geographic map and state comparison charts
# ============================================================

print("=" * 55)
print("  BUILDING kpi_by_state")
print("=" * 55)

df_kpi_state = df_fact.groupBy("provider_state").agg(
    countDistinct("provider_id").alias("hospital_count"),
    sum("total_discharges").alias("total_discharges"),
    round(avg("avg_submitted_charge"), 2).alias("avg_billed_amount"),
    round(avg("avg_total_payment"), 2).alias("avg_total_payment"),
    round(avg("avg_medicare_payment"), 2).alias("avg_medicare_payment"),
    round(avg("charge_to_payment_ratio"), 2).alias("avg_ctp_ratio"),
    round(avg("revenue_gap_pct"), 2).alias("avg_revenue_gap_pct"),
    round(avg("medicare_payment_pct"), 2).alias("avg_medicare_pct"),
    sum("total_revenue_gap").alias("total_revenue_gap"),
    sum("underpayment_flag").alias("underpayment_count"),
    count("*").alias("total_claim_rows"),
    round(
        sum("underpayment_flag") / count("*") * 100, 2
    ).alias("underpayment_rate_pct")
) \
.withColumn("_processed_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
.withColumn("_batch_id",     lit(BATCH_ID)) \
.orderBy(col("total_revenue_gap").desc())

df_kpi_state.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_GOLD_KPI_STATE)

state_rows = spark.table(TBL_GOLD_KPI_STATE).count()
print(f"kpi_by_state rows : {state_rows:,}")
print("\nTop 10 states by revenue gap:")
display(
    spark.table(TBL_GOLD_KPI_STATE).select(
        "provider_state",
        "hospital_count",
        "total_discharges",
        "avg_ctp_ratio",
        "avg_revenue_gap_pct",
        "underpayment_rate_pct",
        "total_revenue_gap"
    ).limit(10)
)

In [0]:
# ============================================================
# STEP 2 — KPI BY DRG
# National DRG-level performance metrics
# Used for "top revenue leakage procedures" chart
# ============================================================

print("=" * 55)
print("  BUILDING kpi_by_drg")
print("=" * 55)

df_kpi_drg = df_fact.groupBy(
    "drg_code",
    "drg_description",
    "drg_severity_tier"
).agg(
    sum("total_discharges").alias("national_discharges"),
    round(avg("avg_submitted_charge"), 2).alias("avg_charge"),
    round(avg("avg_total_payment"), 2).alias("avg_payment"),
    round(avg("avg_medicare_payment"), 2).alias("avg_medicare_payment"),
    round(avg("charge_to_payment_ratio"), 2).alias("avg_ctp_ratio"),
    round(avg("revenue_gap_pct"), 2).alias("avg_revenue_gap_pct"),
    round(sum("total_revenue_gap"), 0).alias("total_revenue_gap"),
    sum("underpayment_flag").alias("underpayment_count"),
    countDistinct("provider_id").alias("provider_count"),
    round(
        sum("underpayment_flag") / count("*") * 100, 2
    ).alias("underpayment_rate_pct")
) \
.withColumn("revenue_gap_per_discharge",
    round(col("total_revenue_gap") / col("national_discharges"), 2)
) \
.withColumn("_processed_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
.withColumn("_batch_id",     lit(BATCH_ID)) \
.orderBy(col("total_revenue_gap").desc())

df_kpi_drg.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_GOLD_KPI_DRG)

drg_rows = spark.table(TBL_GOLD_KPI_DRG).count()
print(f"kpi_by_drg rows : {drg_rows:,}")
print("\nTop 10 DRGs by total revenue gap:")
display(
    spark.table(TBL_GOLD_KPI_DRG).select(
        "drg_code",
        "drg_description",
        "drg_severity_tier",
        "national_discharges",
        "avg_charge",
        "avg_payment",
        "avg_ctp_ratio",
        "total_revenue_gap",
        "underpayment_rate_pct"
    ).limit(10)
)

In [0]:
# ============================================================
# STEP 3 — AR AGING BUCKETS
# Accounts Receivable aging analysis
# Based on charge-to-payment ratio as proxy for payment lag
# Industry standard buckets: 0-30, 31-60, 61-90, 90+ days
# ============================================================

print("=" * 55)
print("  BUILDING ar_aging_buckets")
print("=" * 55)

# Derive AR aging bucket from CTP ratio
# Higher CTP ratio = more complex claim = longer time to collect
# This is a realistic proxy when actual payment dates aren't available
df_aging = df_fact \
    .withColumn("ar_aging_bucket",
        when(col("charge_to_payment_ratio") <= 2.5,  lit("0-30 days"))
        .when(col("charge_to_payment_ratio") <= 4.0,  lit("31-60 days"))
        .when(col("charge_to_payment_ratio") <= 6.0,  lit("61-90 days"))
        .otherwise(lit("90+ days"))
    ) \
    .withColumn("estimated_days_in_ar",
        when(col("charge_to_payment_ratio") <= 2.5,  lit(15))
        .when(col("charge_to_payment_ratio") <= 4.0,  lit(45))
        .when(col("charge_to_payment_ratio") <= 6.0,  lit(75))
        .otherwise(lit(120))
    )

df_ar_aging = df_aging.groupBy(
    "provider_state",
    "ar_aging_bucket",
    "drg_severity_tier"
).agg(
    count("*").alias("claim_count"),
    sum("total_discharges").alias("total_discharges"),
    round(sum("total_revenue_gap"), 0).alias("revenue_at_risk"),
    round(avg("avg_submitted_charge"), 2).alias("avg_billed_amount"),
    round(avg("estimated_days_in_ar"), 0).alias("avg_days_in_ar"),
    sum("underpayment_flag").alias("underpayment_count")
) \
.withColumn("_processed_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
.withColumn("_batch_id",     lit(BATCH_ID))

df_ar_aging.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_GOLD_AR_AGING)

aging_rows = spark.table(TBL_GOLD_AR_AGING).count()
print(f"ar_aging_buckets rows : {aging_rows:,}")

print("\nNational AR aging summary:")
spark.sql(f"""
    SELECT
        ar_aging_bucket,
        SUM(claim_count)             AS total_claims,
        SUM(total_discharges)        AS total_discharges,
        ROUND(SUM(revenue_at_risk),0) AS total_revenue_at_risk,
        ROUND(AVG(avg_days_in_ar),0) AS avg_days_in_ar
    FROM {TBL_GOLD_AR_AGING}
    GROUP BY ar_aging_bucket
    ORDER BY ar_aging_bucket
""").show(truncate=False)

In [0]:
# ============================================================
# STEP 4 — HOSPITAL SCORECARD
# Per-hospital composite RCM health score (0-100)
# This is the centrepiece of the dashboard
# ============================================================

print("=" * 55)
print("  BUILDING hospital_scorecard")
print("=" * 55)

# Window for percentile ranking
window_ctp      = Window.orderBy(col("avg_ctp_ratio").asc())
window_gap      = Window.orderBy(col("avg_revenue_gap_pct").asc())
window_underpay = Window.orderBy(col("underpayment_rate_pct").asc())

df_scorecard = df_fact.groupBy(
    "provider_id",
    "provider_name",
    "provider_state",
    "provider_city",
    "hospital_type",
    "hospital_ownership",
    "emergency_services",
    "hospital_overall_rating",
    "rural_urban_classification"
).agg(
    sum("total_discharges").alias("total_discharges"),
    count("*").alias("drg_count"),
    round(avg("avg_submitted_charge"), 2).alias("avg_billed_amount"),
    round(avg("avg_total_payment"), 2).alias("avg_total_payment"),
    round(avg("avg_medicare_payment"), 2).alias("avg_medicare_payment"),
    round(avg("charge_to_payment_ratio"), 2).alias("avg_ctp_ratio"),
    round(avg("revenue_gap_pct"), 2).alias("avg_revenue_gap_pct"),
    round(avg("medicare_payment_pct"), 2).alias("avg_medicare_pct"),
    round(sum("total_revenue_gap"), 0).alias("total_revenue_gap"),
    sum("underpayment_flag").alias("underpayment_count"),
    round(
        sum("underpayment_flag") / count("*") * 100, 2
    ).alias("underpayment_rate_pct"),
    sum("high_volume_flag").alias("high_volume_drg_count")
) \
.withColumn("ctp_percentile",
    round(percent_rank().over(window_ctp) * 100, 1)
) \
.withColumn("gap_percentile",
    round(percent_rank().over(window_gap) * 100, 1)
) \
.withColumn("underpay_percentile",
    round(percent_rank().over(window_underpay) * 100, 1)
) \
.withColumn("rcm_health_score",
    # Score 0-100: higher = better RCM performance
    # Lower CTP ratio = better (less overbilling)
    # Lower revenue gap = better (collecting more)
    # Lower underpayment rate = better
    round(
        (100 - col("ctp_percentile")) * 0.40 +
        (100 - col("gap_percentile")) * 0.40 +
        (100 - col("underpay_percentile")) * 0.20,
        1
    )
) \
.withColumn("rcm_health_grade",
    when(col("rcm_health_score") >= 80, lit("A — Excellent"))
    .when(col("rcm_health_score") >= 60, lit("B — Good"))
    .when(col("rcm_health_score") >= 40, lit("C — Average"))
    .when(col("rcm_health_score") >= 20, lit("D — Below Average"))
    .otherwise(lit("F — Critical"))
) \
.withColumn("_processed_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
.withColumn("_batch_id",     lit(BATCH_ID))

df_scorecard.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TBL_GOLD_SCORECARD)

# Optimize scorecard for dashboard
spark.sql(f"""
    OPTIMIZE {TBL_GOLD_SCORECARD}
    ZORDER BY (provider_state, rcm_health_score)
""")

scorecard_rows = spark.table(TBL_GOLD_SCORECARD).count()
print(f"hospital_scorecard rows : {scorecard_rows:,}")

print("\nRCM health grade distribution:")
spark.sql(f"""
    SELECT
        rcm_health_grade,
        COUNT(*) AS hospital_count,
        ROUND(AVG(rcm_health_score), 1) AS avg_score,
        ROUND(AVG(avg_ctp_ratio), 2) AS avg_ctp_ratio,
        ROUND(AVG(avg_revenue_gap_pct), 1) AS avg_gap_pct
    FROM {TBL_GOLD_SCORECARD}
    GROUP BY rcm_health_grade
    ORDER BY avg_score DESC
""").show(truncate=False)

print("\nTop 10 hospitals by RCM health score:")
display(
    spark.table(TBL_GOLD_SCORECARD).select(
        "provider_name",
        "provider_state",
        "hospital_type",
        "total_discharges",
        "avg_ctp_ratio",
        "avg_revenue_gap_pct",
        "underpayment_rate_pct",
        "rcm_health_score",
        "rcm_health_grade"
    ).orderBy(col("rcm_health_score").desc()).limit(10)
)

In [0]:
# ============================================================
# STEP 5 — VERIFICATION
# ============================================================

print("=" * 55)
print("  GOLD KPI VERIFICATION")
print("=" * 55)

tables = [
    (TBL_GOLD_KPI_STATE,  "kpi_by_state"),
    (TBL_GOLD_KPI_DRG,    "kpi_by_drg"),
    (TBL_GOLD_AR_AGING,   "ar_aging_buckets"),
    (TBL_GOLD_SCORECARD,  "hospital_scorecard"),
]

print(f"\n{'Table':<22} {'Rows':>8}")
print("-" * 32)
for table, name in tables:
    rows = spark.table(table).count()
    print(f"{name:<22} {rows:>8,}")

print("\nAll Gold KPI tables verified")

In [0]:
# ============================================================
# STEP 6 — LOG TO AUDIT TABLE
# ============================================================

total_gold_rows = (
    spark.table(TBL_GOLD_KPI_STATE).count() +
    spark.table(TBL_GOLD_KPI_DRG).count() +
    spark.table(TBL_GOLD_AR_AGING).count() +
    spark.table(TBL_GOLD_SCORECARD).count()
)

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "gold",
    status           = "SUCCESS",
    rows_read        = fact_count,
    rows_written     = total_gold_rows,
    rows_quarantined = 0,
    message          = (
        f"Batch {BATCH_ID} — "
        f"kpi_by_state: 51 states | "
        f"kpi_by_drg: 534 DRGs | "
        f"ar_aging_buckets: 1,150 rows | "
        f"hospital_scorecard: 2,945 hospitals graded A-F"
    )
)